<a href="https://www.kaggle.com/code/tommasofacchin/01-resident-evil-data-scraping-and-generation?scriptVersionId=306170894" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np 
import pandas as pd 
import requests
import re
from pathlib import Path
from bs4 import BeautifulSoup

# Resident Evil: data scraping and generation

Resident Evil game universe dataset created by web scraping and AI generation, intended as a base for future projects.

# Datesets

**Game** *(id, title, year, type, chronology_order)*: AI Generated

**Character** *(id, name, role)*: Resident evil 4 characters scraped from wikipedia, the rest AI generated.

**GameAppearence** *(game_id, character_id, role)*

**Dialogues** *(id, game_id, scene_id, line_index, character_id, text, timestamp, source)*: scraped from game transcripts.

**Scenes** *(id, game_id, name, sequence_index)*: scraped from game transcripts.


## Games

In [2]:
games_data = [
    # id, title, year, type, chronology_order
    ("re",     "Resident Evil",                             1996, "mainline", 2),
    ("re2",    "Resident Evil 2",                           1998, "mainline", 4),
    ("re3",    "Resident Evil 3: Nemesis",                  1999, "mainline", 3),
    ("re_ve",  "Resident Evil: Code – Veronica",            2000, "mainline", 5),
    ("re0",    "Resident Evil 0",                           2002, "mainline", 1),
    ("re4",    "Resident Evil 4",                           2005, "mainline", 6),
    ("re5",    "Resident Evil 5",                           2009, "mainline", 8),
    ("re_re",  "Resident Evil: Revelations",                2012, "spinoff",  7),
    ("re6",    "Resident Evil 6",                           2012, "mainline", 10),
    ("re_re2", "Resident Evil: Revelations 2",              2015, "spinoff",  9),
    ("re7",    "Resident Evil 7: Biohazard",                2017, "mainline", 11),
    ("re8",    "Resident Evil Village",                     2021, "mainline", 12),
    ("re_sr",  "Resident Evil Village – Shadow of Rose",    2022, "spinoff",  13),
    ("re9",    "Resident Evil Requiem",                     2026, "mainline", 14),
]

games = pd.DataFrame(
    games_data,
    columns=["id", "title", "year", "type", "chronology_order"]
)

games.to_csv("games.csv", index=False)
games.head(20)

,id,title,year,type,chronology_order
0,re,Resident Evil,1996,mainline,2
1,re2,Resident Evil 2,1998,mainline,4
2,re3,Resident Evil 3: Nemesis,1999,mainline,3
3,re_ve,Resident Evil: Code – Veronica,2000,mainline,5
4,re0,Resident Evil 0,2002,mainline,1
5,re4,Resident Evil 4,2005,mainline,6
6,re5,Resident Evil 5,2009,mainline,8
7,re_re,Resident Evil: Revelations,2012,spinoff,7
8,re6,Resident Evil 6,2012,mainline,10
9,re_re2,Resident Evil: Revelations 2,2015,spinoff,9


## Characters and GameAppearence

In [3]:
characters = pd.DataFrame(
    columns=[
        "id",       
        "name",    
        "role",     
    ]
)

gameAppearance = pd.DataFrame(
    columns=[
        "game_id",       
        "character_id", 
        "role",        
    ]
)

In [4]:
# Resident evil 4 (2005): easy scraping from wikipedia
game_id = "re4"
URL = "https://it.wikipedia.org/wiki/Resident_Evil_4"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# Patch: Kaggle runs notebook offline during save and run, so i downloaded the wikipedia page in order for it to work.
html_path = "/kaggle/input/datasets/tommasofacchin/resident-evil-4-wikipedia/Resident Evil 4 - Wikipedia.html" 

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
# -- end patch


names = []
roles = [] 
current_role = None 

for tag in soup.find_all(["h2", "h3"]):
    if tag.name == "h2":
        text = tag.get_text(strip=True)
        print("H2:", text)  

        if text == "Personaggi":
            current_role = "hero"
        elif text == "Nemici":
            current_role = "villain"
        else:
            current_role = None

    elif tag.name == "h3" and current_role is not None:
        text = tag.get_text(strip=True)

        names.append(text)
        roles.append(current_role)
        print("  H3:", text)

print("\n# characters:", len(names))
print(list(zip(names, roles)))


re4_df = pd.DataFrame({
    "name": names,
    "role_in_game": roles,  
})

# Characters
re4_df.loc[re4_df["name"] == "Albert Wesker", "role_in_game"] = "villain"
existing_names = set(characters["name"])

missing_mask = ~re4_df["name"].isin(existing_names)
missing_rows = re4_df[missing_mask]

next_id = characters["id"].max() + 1 if len(characters) > 0 else 1

new_rows = []
for _, row in missing_rows.iterrows():
    new_rows.append({
        "id": next_id,
        "name": row["name"],
        "role": row["role_in_game"],
    })
    next_id += 1

if new_rows:
    new_chars = pd.DataFrame(new_rows)
    characters = pd.concat([characters, new_chars], ignore_index=True)

# Game appearences
rows = []
for _, row in re4_df.iterrows():
    name = row["name"]
    role_in_game = row["role_in_game"]

    match = characters[characters["name"] == name]
    if match.empty:
        continue
    cid = match["id"].iloc[0]

    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re4 = pd.DataFrame(rows)

gameAppearance = pd.concat([gameAppearance, gameApp_re4], ignore_index=True)

print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")

H2: Indice
H2: Trama
H2: Modalità di gioco
H2: Personaggi
  H3: Leon Scott Kennedy
  H3: Ashley Graham
  H3: Ada Wong
  H3: Ingrid Hunnigan
  H3: Luis Sera
  H3: Albert Wesker
  H3: HUNK
  H3: Mercante d'armi
H2: Nemici
  H3: Las Plagas
  H3: Los Ganados
  H3: Lord Osmund Saddler
  H3: Ramon Salazar
  H3: Bitores Mendez
  H3: Jack Krauser
H2: Versioni
H2: Confronti territoriali
H2: Versione beta
H2: Remake
H2: Edizione VR
H2: Accoglienza
H2: Doppiaggio
H2: Colonna sonora
H2: Note
H2: Altri progetti
H2: Collegamenti esterni

# characters: 14
[('Leon Scott Kennedy', 'hero'), ('Ashley Graham', 'hero'), ('Ada Wong', 'hero'), ('Ingrid Hunnigan', 'hero'), ('Luis Sera', 'hero'), ('Albert Wesker', 'hero'), ('HUNK', 'hero'), ("Mercante d'armi", 'hero'), ('Las Plagas', 'villain'), ('Los Ganados', 'villain'), ('Lord Osmund Saddler', 'villain'), ('Ramon Salazar', 'villain'), ('Bitores Mendez', 'villain'), ('Jack Krauser', 'villain')]


characters:     id                 name     role
0    1   Leon

In [5]:
# Resident evil (1996): AI generated
game_id = "re" 
next_id = characters["id"].max() + 1 

character_data = [
    # name,                 main_alignment
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "hero"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]
existing_names = set(characters["name"])

new_rows = []
for name, role in character_data:
    if name in existing_names:           
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role, 
    })
    next_id += 1

new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])

characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))
gameAppearance_data = [
    # name,                 role_in_game
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "support"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re1 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re1], ignore_index=True)


print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")



characters:     id                 name     role
0    1   Leon Scott Kennedy     hero
1    2        Ashley Graham     hero
2    3             Ada Wong     hero
3    4      Ingrid Hunnigan     hero
4    5            Luis Sera     hero
5    6        Albert Wesker  villain
6    7                 HUNK     hero
7    8      Mercante d'armi     hero
8    9           Las Plagas  villain
9   10          Los Ganados  villain
10  11  Lord Osmund Saddler  villain
11  12        Ramon Salazar  villain
12  13       Bitores Mendez  villain
13  14         Jack Krauser  villain
14  15       Chris Redfield     hero
15  16       Jill Valentine     hero
16  17         Barry Burton     hero
17  18         Brad Vickers  support
18  19         Joseph Frost  support
19  20     Rebecca Chambers  support
20  21        Richard Aiken  support
21  22        Enrico Marini  support
22  23        Forest Speyer  support
23  24  Kenneth J. Sullivan  support


Game appearence:    game_id character_id     role
0      re

In [6]:
# Resident evil 2 (1998): AI generated
game_id = "re2"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),      
    ("Claire Redfield",    "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue 
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),       
    ("Claire Redfield",    "hero"),        
    ("Ada Wong",           "support"),     
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re2 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re2], ignore_index=True)
print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == 2])

    id                name     role
0    1  Leon Scott Kennedy     hero
2    3            Ada Wong     hero
24  25     Claire Redfield     hero
25  26       Sherry Birkin  support
26  27      William Birkin  villain
27  28      Annette Birkin  support
28  29         Brian Irons  villain
29  30      Marvin Branagh  support
30  31      Ben Bertolucci  support
31  32        Robert Kendo  support
Empty DataFrame
Columns: [game_id, character_id, role]
Index: []


In [7]:
# Resident evil 3: Nemesis (1999): AI generated
game_id = "re3"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),      
    ("Carlos Oliveira",  "hero"),
    ("Nemesis",          "villain"),
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),   
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),       
    ("Carlos Oliveira",  "hero"),       
    ("Nemesis",          "villain"), 
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re3 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re3], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == 3])

    id              name     role
15  16    Jill Valentine     hero
17  18      Brad Vickers  support
29  30    Marvin Branagh  support
32  33   Carlos Oliveira     hero
33  34           Nemesis  villain
34  35  Nikolai Zinoviev  villain
35  36    Mikhail Victor  support
36  37       Dario Rosso  support
Empty DataFrame
Columns: [game_id, character_id, role]
Index: []


In [8]:
# Resident evil: Code – Veronica (2000): AI generated
game_id = "re_ve"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Claire Redfield",  "hero"),
    ("Steve Burnside",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Albert Wesker",    "villain"),
    ("Alexia Ashford",   "villain"),
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Claire Redfield",  "hero"),        
    ("Steve Burnside",   "hero"),       
    ("Chris Redfield",   "hero"),       
    ("Albert Wesker",    "villain"), 
    ("Alexia Ashford",   "villain"), 
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_cvx = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_cvx], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id                name     role
5    6       Albert Wesker  villain
14  15      Chris Redfield     hero
24  25     Claire Redfield     hero
37  38      Steve Burnside     hero
38  39      Alexia Ashford  villain
39  40      Alfred Ashford  villain
40  41  Rodrigo Juan Raval  support
   game_id character_id     role
43   re_ve           25     hero
44   re_ve           38     hero
45   re_ve           15     hero
46   re_ve            6  villain
47   re_ve           39  villain
48   re_ve           40  villain
49   re_ve           41  support


In [9]:
# Resident evil 0 (2002): AI generated
game_id = "re0"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rebecca Chambers", "hero"),
    ("Billy Coen",       "hero"),
    ("James Marcus",     "villain"),
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rebecca Chambers", "hero"),       
    ("Billy Coen",       "hero"),        
    ("James Marcus",     "villain"),  
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re0 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re0], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])


    id              name     role
5    6     Albert Wesker  villain
19  20  Rebecca Chambers  support
21  22     Enrico Marini  support
26  27    William Birkin  villain
41  42        Billy Coen     hero
42  43      James Marcus  villain
43  44      Edward Dewey  support
   game_id character_id     role
50     re0           20     hero
51     re0           42     hero
52     re0           43  villain
53     re0           44  support
54     re0           22  support
55     re0            6  villain
56     re0           27  villain


In [10]:
# Resident evil 5 (2009): AI generated
game_id = "re5"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Chris Redfield",  "hero"),
    ("Sheva Alomar",    "hero"),
    ("Albert Wesker",   "villain"),
    ("Jill Valentine",  "hero"),
    ("Ricardo Irving",  "villain"),
    ("Excella Gionne",  "villain"),
    ("Josh Stone",      "support"),
    ("Ozwell E. Spencer","villain"),
    ("Kirk Mathison", "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Chris Redfield",   "hero"),        
    ("Sheva Alomar",     "hero"),        
    ("Albert Wesker",    "villain"),  
    ("Jill Valentine",   "support"),     
    ("Ricardo Irving",   "villain"),
    ("Excella Gionne",   "villain"),
    ("Josh Stone",       "support"),
    ("Ozwell E. Spencer","support"),     
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re5 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re5], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id               name     role
5    6      Albert Wesker  villain
14  15     Chris Redfield     hero
15  16     Jill Valentine     hero
44  45       Sheva Alomar     hero
45  46     Ricardo Irving  villain
46  47     Excella Gionne  villain
47  48         Josh Stone  support
48  49  Ozwell E. Spencer  villain
49  50      Kirk Mathison  support
   game_id character_id     role
57     re5           15     hero
58     re5           45     hero
59     re5            6  villain
60     re5           16  support
61     re5           46  villain
62     re5           47  villain
63     re5           48  support
64     re5           49  support


In [11]:
# Resident Evil: Revelations (2012): AI generated
game_id = "re_re"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "hero"),
    ("Jessica Sherawat", "hero"),    
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),  
    ("Morgan Lansdale",  "villain"),
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "support"),
    ("Jessica Sherawat", "support"),
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),    
    ("Morgan Lansdale",  "villain"), 
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_rev = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id              name     role
14  15    Chris Redfield     hero
15  16    Jill Valentine     hero
50  51    Parker Luciani     hero
51  52  Jessica Sherawat     hero
52  53  Clive R. O'Brian  support
53  54    Raymond Vester  support
54  55   Morgan Lansdale  villain
55  56       Jack Norman  villain
56  57      Rachel Foley  support
57  58      Keith Lumley  support
58  59     Quint Cetcham  support
   game_id character_id     role
65   re_re           16     hero
66   re_re           15     hero
67   re_re           51  support
68   re_re           52  support
69   re_re           53  support
70   re_re           54  support
71   re_re           55  villain
72   re_re           56  villain
73   re_re           57  support
74   re_re           58  support
75   re_re           59  support


In [12]:
# Resident evil 6 (2012): AI generated
game_id = "re6"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),
    ("Chris Redfield",     "hero"),
    ("Jake Muller",        "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("Helena Harper",      "hero"),
    ("Piers Nivans",       "hero"),
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),        
    ("Helena Harper",      "hero"),        
    ("Chris Redfield",     "hero"),       
    ("Piers Nivans",       "hero"),        
    ("Jake Muller",        "hero"),       
    ("Sherry Birkin",      "support"),    
    ("Ada Wong",           "support"),    
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re6 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re6], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id                name     role
0    1  Leon Scott Kennedy     hero
2    3            Ada Wong     hero
3    4     Ingrid Hunnigan     hero
14  15      Chris Redfield     hero
25  26       Sherry Birkin  support
59  60         Jake Muller     hero
60  61       Helena Harper     hero
61  62        Piers Nivans     hero
62  63    Derek C. Simmons  villain
   game_id character_id     role
76     re6            1     hero
77     re6           61     hero
78     re6           15     hero
79     re6           62     hero
80     re6           60     hero
81     re6           26  support
82     re6            3  support
83     re6            4  support
84     re6           63  villain


In [13]:
# Resident Evil: Revelations 2 (2015): AI generated
game_id = "re_re2"
next_id = characters["id"].max() + 1 

character_data = [
    # name,            main_alignment
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,            role_in_game
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_rev2 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev2], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id             name     role
16  17     Barry Burton     hero
24  25  Claire Redfield     hero
63  64     Moira Burton  support
64  65    Natalia Korda  support
65  66      Alex Wesker  villain
   game_id character_id     role
85  re_re2           25     hero
86  re_re2           17     hero
87  re_re2           64  support
88  re_re2           65  support
89  re_re2           66  villain


In [14]:
# Resident Evil 7: Biohazard (2017): AI generated
game_id = "re7"
next_id = characters["id"].max() + 1 

character_data = [
    # name,           main_alignment
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "hero"),
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,           role_in_game
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "support"),     
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_re7 = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re7], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id              name     role
66  67     Ethan Winters     hero
67  68       Mia Winters     hero
68  69        Jack Baker  villain
69  70  Marguerite Baker  villain
70  71       Lucas Baker  villain
71  72         Zoe Baker  support
72  73           Eveline  villain
   game_id character_id     role
90     re7           67     hero
91     re7           68  support
92     re7           69  villain
93     re7           70  villain
94     re7           71  villain
95     re7           72  support
96     re7           73  villain


In [15]:
# Resident Evil Village (2021): AI generated
game_id = "re8"
next_id = characters["id"].max() + 1 

character_data = [
    # name,               main_alignment
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "hero"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "hero"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,               role_in_game
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "support"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "support"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_village = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_village], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id               name     role
14  15     Chris Redfield     hero
66  67      Ethan Winters     hero
67  68        Mia Winters     hero
73  74   Rosemary Winters  support
74  75     Mother Miranda  villain
75  76  Alcina Dimitrescu  villain
76  77    Karl Heisenberg  villain
77  78   Salvatore Moreau  villain
78  79   Donna Beneviento  villain
    game_id character_id     role
97      re8           67     hero
98      re8           68  support
99      re8           74  support
100     re8           15  support
101     re8           75  villain
102     re8           76  villain
103     re8           77  villain
104     re8           78  villain
105     re8           79  villain


In [16]:
# Resident Evil Village – Shadow of Rose (2022): AI generated
game_id = "re_sr"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "hero"),
    ("Mother Miranda",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "support"),
    ("Mother Miranda",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_sor = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_sor], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id              name     role
66  67     Ethan Winters     hero
73  74  Rosemary Winters  support
74  75    Mother Miranda  villain
    game_id character_id     role
106   re_sr           74     hero
107   re_sr           67  support
108   re_sr           75  villain


In [17]:
# Resident Evil Requiem (2026): AI generated
game_id = "re9"
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Grace Ashcroft",   "hero"),
    ("Leon Scott Kennedy","hero"),
    ("Sherry Birkin",    "support"),
    ("Victor Gideon",    "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Grace Ashcroft",    "hero"),       
    ("Leon Scott Kennedy","hero"),       
    ("Sherry Birkin",     "support"),
    ("Victor Gideon",     "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support")
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "game_id": game_id,
        "character_id": cid,
        "role": role_in_game,
    })

gameApp_requiem = pd.DataFrame(rows, columns=["game_id", "character_id", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_requiem], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["game_id"] == game_id])

    id                name     role
0    1  Leon Scott Kennedy     hero
25  26       Sherry Birkin  support
79  80      Grace Ashcroft     hero
80  81       Victor Gideon  villain
81  82      Nathan Dempsey  support
82  83     Alyssa Ashcroft  support
    game_id character_id     role
109     re9           80     hero
110     re9            1     hero
111     re9           26  support
112     re9           81  villain
113     re9           82  support
114     re9           83  support


In [18]:
characters.to_csv("characters.csv", index=False)
print(characters.to_string())

    id                 name     role
0    1   Leon Scott Kennedy     hero
1    2        Ashley Graham     hero
2    3             Ada Wong     hero
3    4      Ingrid Hunnigan     hero
4    5            Luis Sera     hero
5    6        Albert Wesker  villain
6    7                 HUNK     hero
7    8      Mercante d'armi     hero
8    9           Las Plagas  villain
9   10          Los Ganados  villain
10  11  Lord Osmund Saddler  villain
11  12        Ramon Salazar  villain
12  13       Bitores Mendez  villain
13  14         Jack Krauser  villain
14  15       Chris Redfield     hero
15  16       Jill Valentine     hero
16  17         Barry Burton     hero
17  18         Brad Vickers  support
18  19         Joseph Frost  support
19  20     Rebecca Chambers  support
20  21        Richard Aiken  support
21  22        Enrico Marini  support
22  23        Forest Speyer  support
23  24  Kenneth J. Sullivan  support
24  25      Claire Redfield     hero
25  26        Sherry Birkin  support
2

In [19]:
gameAppearance = (
    gameAppearance
    .sort_values(by=["game_id", "character_id"])
    .reset_index(drop=True)
)

# for semplicity
gameAppearance = (
    gameAppearance
    .merge(
        games[["id", "title"]],
        left_on="game_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"]) 
    .rename(columns={"title": "game_title"})
    
    .merge(
        characters[["id", "name"]],
        left_on="character_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"]) 
    .rename(columns={"name": "character_name"})
)

gameAppearance.to_csv("gameAppearance.csv", index=False)
print(gameAppearance.to_string())

    game_id character_id     role                              game_title       character_name
0        re            6  villain                           Resident Evil        Albert Wesker
1        re           15     hero                           Resident Evil       Chris Redfield
2        re           16     hero                           Resident Evil       Jill Valentine
3        re           17  support                           Resident Evil         Barry Burton
4        re           18  support                           Resident Evil         Brad Vickers
5        re           19  support                           Resident Evil         Joseph Frost
6        re           20  support                           Resident Evil     Rebecca Chambers
7        re           21  support                           Resident Evil        Richard Aiken
8        re           22  support                           Resident Evil        Enrico Marini
9        re           23  support                 

## Scenes and interactions

In [20]:
scenes = pd.DataFrame(columns=["game_id", "scene_id", "title", "source"])
interactions = pd.DataFrame(columns=["game_id", "scene_id", "character_id"])

In [21]:
game_id = "re"
source = "https://gamefaqs.gamespot.com/ps/198462-resident-evil-directors-cut/faqs/20010"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/resident_evil_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^Scene\s+(\d+)\.\s*(.+)", re.IGNORECASE)
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}

current_scene_id = None
scene_counter = 0

known_characters = {
    "Chris": "Chris Redfield",
    "Chris Redfield": "Chris Redfield",
    "Jill": "Jill Valentine",
    "Jill Valentine": "Jill Valentine",
    "Barry": "Barry Burton",
    "Barry Burton": "Barry Burton",
    "Rebecca": "Rebecca Chambers",
    "Rebecca Chambers": "Rebecca Chambers",
    "Wesker": "Albert Wesker",
    "Albert Wesker": "Albert Wesker",
    "Brad": "Brad Vickers",
    "Brad Vickers": "Brad Vickers",
}


char_name_to_id = dict(zip(characters["name"], characters["id"]))

canon_to_id = {k: char_name_to_id[k] for k in known_characters.values() if k in char_name_to_id}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        title = m_scene.group(2).strip()
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:02d}"
        current_scene_id = scene_id
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_key = speaker_raw.strip(" *[]").strip()
            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)
print(scenes_new.shape)
print(interactions_new.shape)

(36, 4)
(69, 3)


In [22]:
game_id = "re2"
source = "https://gamefaqs.gamespot.com/ps/198458-resident-evil-2/faqs/32821"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_2_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^Scene\s+(\d+):\s*(.+)", re.IGNORECASE)
characters_line_re = re.compile(r"^Characters:\s*(.+)", re.IGNORECASE)

scene_chars = {}
scenes_rows = []

current_scene_id = None
scene_counter = 0

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        title = m_scene.group(2).strip()
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:02d}"

        current_scene_id = scene_id
        scenes_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    m_chars = characters_line_re.match(line)
    if m_chars and current_scene_id is not None:
        chars_str = m_chars.group(1).rstrip(".")
        parts = [c.strip() for c in chars_str.split(",") if c.strip()]
        for name in parts:
            scene_chars[current_scene_id].add(name)
        continue

id_to_transcript = {
    1:  "Leon Kennedy",
    3:  "Ada Wong",
    25: "Claire Redfield",
    26: "Sherry Birkin",
    27: "William Birkin",
    28: "Annette Birkin",
    31: "Ben Bertolucci",
}

transcript_to_id = {v: k for k, v in id_to_transcript.items()}

scenes_new = pd.DataFrame(scenes_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, names in scene_chars.items():
    for name in names:
        parts = [p.strip() for p in name.split(" and ") if p.strip()]
        for p in parts:
            cid = transcript_to_id.get(p)
            if cid is not None:
                interaction_rows.append({
                    "game_id": game_id,
                    "scene_id": scene_id,
                    "character_id": cid,
                })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)
print(scenes_new.shape)
print(interactions_new.shape)

(80, 4)
(177, 3)


In [23]:
game_id = "re3"
source = "https://gamefaqs.gamespot.com/ps/198459-resident-evil-3-nemesis/faqs/60207"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_3_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^(\d+)\.\s*(.+)")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Jill": "Jill Valentine",
    "Jill Valentine": "Jill Valentine",
    "Carlos": "Carlos Oliveira",
    "Carlos Oliveira": "Carlos Oliveira",
    "Nemesis": "Nemesis",
    "Nicholai": "Nikolai Zinoviev",
    "Nikolai": "Nikolai Zinoviev",
    "Mikhail": "Mikhail Victor",
    "Mikhail Victor": "Mikhail Victor",
    "Brad": "Brad Vickers",
    "Brad Vickers": "Brad Vickers",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        title = m_scene.group(2).strip()
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:02d}"
        current_scene_id = scene_id

        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    # match dialogue lines
    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()
            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(180, 4)
(52, 3)
  game_id scene_id                    title  \
0     re3   re3_01  Raccoon City’s Narrator   
1     re3   re3_02          Defense Crushed   
2     re3   re3_03               The Escape   
3     re3   re3_04               Surrounded   
4     re3   re3_05                   Choice   

                                              source  
0  https://gamefaqs.gamespot.com/ps/198459-reside...  
1  https://gamefaqs.gamespot.com/ps/198459-reside...  
2  https://gamefaqs.gamespot.com/ps/198459-reside...  
3  https://gamefaqs.gamespot.com/ps/198459-reside...  
4  https://gamefaqs.gamespot.com/ps/198459-reside...  
  game_id scene_id  character_id
0     re3   re3_91            16
1     re3   re3_95            16
2     re3   re3_96            18
3     re3   re3_97            16
4     re3   re3_97            18


In [24]:
game_id = "re_ve"
source = "https://gamefaqs.gamespot.com/gamecube/535839-resident-evil-code-veronica-x/faqs/41999"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_veronica_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

section_re = re.compile(r"^\s*\d\.\d\.\d\.\s*(.+)$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Claire": "Claire Redfield",
    "CLAIRE": "Claire Redfield",
    "Steve": "Steve Burnside",
    "STEVE": "Steve Burnside",
    "Chris": "Chris Redfield",
    "CHRIS": "Chris Redfield",
    "Wesker": "Albert Wesker",
    "WESKER": "Albert Wesker",
    "Alexia": "Alexia Ashford",
    "ALEXIA": "Alexia Ashford",
    "Alfred": "Alfred Ashford",
    "ALFRED": "Alfred Ashford",
    "Rodrigo": "Rodrigo Juan Raval",
    "RODRIGO": "Rodrigo Juan Raval",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.rstrip("\n")
    stripped = line.strip()
    if not stripped:
        continue

    m_sec = section_re.match(stripped)
    if m_sec:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id
        title = stripped

        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(stripped)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })
interactions_new = pd.DataFrame(interaction_rows)

scenes = pd.concat([scenes, scenes_new], ignore_index=True)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(34, 4)
(70, 3)
  game_id   scene_id                         title  \
0   re_ve  re_ve_001    3.0.0. Rodrigo Has a Heart   
1   re_ve  re_ve_002     3.0.1. Love at First Shot   
2   re_ve  re_ve_003       3.0.2.\tContacting Leon   
3   re_ve  re_ve_004    3.0.3. Double Lugar Danger   
4   re_ve  re_ve_005  3.0.4.   Woo Pretty Red Dot!   

                                              source  
0  https://gamefaqs.gamespot.com/gamecube/535839-...  
1  https://gamefaqs.gamespot.com/gamecube/535839-...  
2  https://gamefaqs.gamespot.com/gamecube/535839-...  
3  https://gamefaqs.gamespot.com/gamecube/535839-...  
4  https://gamefaqs.gamespot.com/gamecube/535839-...  
  game_id   scene_id  character_id
0   re_ve  re_ve_001            41
1   re_ve  re_ve_001            25
2   re_ve  re_ve_002            25
3   re_ve  re_ve_002            38
4   re_ve  re_ve_003            25


In [25]:
game_id = "re0"
source = "https://gamefaqs.gamespot.com/gamecube/198460-resident-evil-0/faqs/20253"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_0_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r'^\*{3}(.+)\*{3}$')
dialogue_line_re = re.compile(r'^([^:\[]+?):\s*(.+)$')

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Rebecca": "Rebecca Chambers",
    "REBECCA": "Rebecca Chambers",
    "Billy": "Billy Coen",
    "BILLY": "Billy Coen",
    "Enrico": "Enrico Marini",
    "ENRICO": "Enrico Marini",
    "Edward": "Edward Dewey",
    "EDWARD": "Edward Dewey",
    "Narrator": "James Marcus",
    "NARRATOR": "James Marcus",
    "Marcus": "James Marcus",
    "MARCUS": "James Marcus",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.rstrip("\n")
    stripped = line.strip()
    if not stripped:
        continue

    m_scene = scene_line_re.match(stripped)
    if m_scene:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id
        title = m_scene.group(1).strip()

        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(stripped)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
valid_scene_ids = {sid for sid, ids in scene_chars.items() if ids}
scenes_new = scenes_new[scenes_new["scene_id"].isin(valid_scene_ids)].reset_index(drop=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    if scene_id not in valid_scene_ids:
        continue
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })
interactions_new = pd.DataFrame(interaction_rows)

scenes = pd.concat([scenes, scenes_new], ignore_index=True)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(36, 4)
(66, 3)
  game_id scene_id                       title  \
0     re0  re0_012                    Prologue   
1     re0  re0_013  Rebecca Searches the Train   
2     re0  re0_014       When the Dust Settles   
3     re0  re0_015               Meet Lt. Coen   
4     re0  re0_016      Edward's Final Moments   

                                              source  
0  https://gamefaqs.gamespot.com/gamecube/198460-...  
1  https://gamefaqs.gamespot.com/gamecube/198460-...  
2  https://gamefaqs.gamespot.com/gamecube/198460-...  
3  https://gamefaqs.gamespot.com/gamecube/198460-...  
4  https://gamefaqs.gamespot.com/gamecube/198460-...  
  game_id scene_id  character_id
0     re0  re0_012            44
1     re0  re0_012            20
2     re0  re0_012            22
3     re0  re0_013            20
4     re0  re0_014            20


In [26]:
game_id = "re4"
source = "https://gamefaqs.gamespot.com/gamecube/535840-resident-evil-4/faqs/34828" 
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_4_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^(Chapter\s*\d+-\d+|Prologue|Epilogue|.+?Scene\s*\d+|.+?Cutscene\s*\d+|.+?Chapter\s*\d+-\d+)$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Leon": "Leon Scott Kennedy",
    "Leon S. Kennedy": "Leon Scott Kennedy",
    "Ashley": "Ashley Graham",
    "Ashley Graham": "Ashley Graham",
    "Ada": "Ada Wong",
    "Ada Wong": "Ada Wong",
    "Hunnigan": "Ingrid Hunnigan",
    "Ingrid Hunnigan": "Ingrid Hunnigan",
    "Luis": "Luis Sera",
    "Luis Sera": "Luis Sera",
    "Saddler": "Lord Osmund Saddler",
    "Osmund Saddler": "Lord Osmund Saddler",
    "Salazar": "Ramon Salazar",
    "Ramon Salazar": "Ramon Salazar",
    "Mendez": "Bitores Mendez",
    "Bitores Mendez": "Bitores Mendez",
    "Krauser": "Jack Krauser",
    "Jack Krauser": "Jack Krauser",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        title = m_scene.group(1).strip()
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(18, 4)
(46, 3)
  game_id scene_id        title  \
0     re4  re4_001  Chapter 1-1   
1     re4  re4_002  Chapter 1-2   
2     re4  re4_003  Chapter 1-3   
3     re4  re4_004  Chapter 2-1   
4     re4  re4_005  Chapter 2-2   

                                              source  
0  https://gamefaqs.gamespot.com/gamecube/535840-...  
1  https://gamefaqs.gamespot.com/gamecube/535840-...  
2  https://gamefaqs.gamespot.com/gamecube/535840-...  
3  https://gamefaqs.gamespot.com/gamecube/535840-...  
4  https://gamefaqs.gamespot.com/gamecube/535840-...  
  game_id scene_id  character_id
0     re4  re4_001             1
1     re4  re4_002             1
2     re4  re4_002            13
3     re4  re4_002             5
4     re4  re4_003             1


In [27]:
game_id = "re5"
source = "https://gamefaqs.gamespot.com/xbox360/929197-resident-evil-5/faqs/56254" 
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_5_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^\{CUTSCENE\s*#(\d+)\s*--\s*(.+)\}$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "CHRIS": "Chris Redfield",
    "Chris": "Chris Redfield",
    "CHRIS REDFIELD": "Chris Redfield",
    "SHEVA": "Sheva Alomar",
    "Sheva": "Sheva Alomar",
    "SHEVA ALOMAR": "Sheva Alomar",
    "JOSH": "Josh Stone",
    "Josh": "Josh Stone",
    "JOSH STONE": "Josh Stone",
    "KIRK": "Kirk Mathison",   
    "Kirk": "Kirk Mathison",
    "WESKER": "Albert Wesker",
    "Wesker": "Albert Wesker",
    "IRVING": "Ricardo Irving",
    "Irving": "Ricardo Irving",
    "RICARDO IRVING": "Ricardo Irving",
    "EXCELLA": "Excella Gionne",
    "Excella": "Excella Gionne",
    "SOLDIER": "Los Ganados",  
    "MAJINI": "Los Ganados",
    "JILL": "Jill Valentine",
    "JILL VALENTINE": "Jill Valentine",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        num = int(m_scene.group(1))
        title = m_scene.group(2).strip()

        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(48, 4)
(127, 3)
  game_id scene_id                title  \
0     re5  re5_001    Welcome to Africa   
1     re5  re5_002            Magic Act   
2     re5  re5_003  The Butcher, Part 1   
3     re5  re5_004  The Butcher, Part 2   
4     re5  re5_005      First Encounter   

                                              source  
0  https://gamefaqs.gamespot.com/xbox360/929197-r...  
1  https://gamefaqs.gamespot.com/xbox360/929197-r...  
2  https://gamefaqs.gamespot.com/xbox360/929197-r...  
3  https://gamefaqs.gamespot.com/xbox360/929197-r...  
4  https://gamefaqs.gamespot.com/xbox360/929197-r...  
  game_id scene_id  character_id
0     re5  re5_001            10
1     re5  re5_001            50
2     re5  re5_001            45
3     re5  re5_001            15
4     re5  re5_003            15


In [28]:
game_id = "re6"
source = "https://game-scripts-wiki.blogspot.com/2021/06/resident-evil-6-transcript.html" 
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_6_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^Chapter\s+(\d+)", re.IGNORECASE)
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Leon": "Leon Scott Kennedy",
    "LEON": "Leon Scott Kennedy",
    "Leon S. Kennedy": "Leon Scott Kennedy",
    "Helena": "Helena Harper",
    "HELENA": "Helena Harper",
    "Chris": "Chris Redfield",
    "CHRIS": "Chris Redfield",
    "Piers": "Piers Nivans",
    "PIERS": "Piers Nivans",
    "Jake": "Jake Muller",
    "JAKE": "Jake Muller",
    "Sherry": "Sherry Birkin",
    "SHERRY": "Sherry Birkin",
    "Ada": "Ada Wong",
    "ADA": "Ada Wong",
    "Simmons": "Derek C. Simmons",
    "SIMMONS": "Derek C. Simmons",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        chap_num = int(m_scene.group(1))

        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        title = f"Chapter {chap_num}"
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(10, 4)
(39, 3)
  game_id scene_id      title  \
0     re6  re6_001  Chapter 1   
1     re6  re6_002  Chapter 2   
2     re6  re6_003  Chapter 3   
3     re6  re6_004  Chapter 4   
4     re6  re6_005  Chapter 5   

                                              source  
0  https://game-scripts-wiki.blogspot.com/2021/06...  
1  https://game-scripts-wiki.blogspot.com/2021/06...  
2  https://game-scripts-wiki.blogspot.com/2021/06...  
3  https://game-scripts-wiki.blogspot.com/2021/06...  
4  https://game-scripts-wiki.blogspot.com/2021/06...  
  game_id scene_id  character_id
0     re6  re6_001            26
1     re6  re6_001            60
2     re6  re6_001            62
3     re6  re6_001            15
4     re6  re6_002            26


In [29]:
game_id = "re7"
source = "https://game-scripts-wiki.blogspot.com/2018/12/resident-evil-7-full-transcript.html"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_7_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^\[.+\]$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Ethan": "Ethan Winters",
    "ETHAN": "Ethan Winters",
    "Mia": "Mia Winters",
    "MIA": "Mia Winters",
    "Jack": "Jack Baker",
    "JACK": "Jack Baker",
    "Marguerite": "Marguerite Baker",
    "MARGUERITE": "Marguerite Baker",
    "Lucas": "Lucas Baker",
    "LUCAS": "Lucas Baker",
    "Zoe": "Zoe Baker",
    "ZOE": "Zoe Baker",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        title = line.strip("[]").strip()
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(227, 4)
(117, 3)
  game_id scene_id                                              title  \
0     re7  re7_001      You see someone watching a video on a laptop.   
1     re7  re7_002  Then you see how the webcam on your laptop tur...   
2     re7  re7_003  The scene is changing. Ethan drives his car al...   
3     re7  re7_004  Ethan comes to the right house in the middle o...   
4     re7  re7_005  Ethan sees that the gate to the house is locke...   

                                              source  
0  https://game-scripts-wiki.blogspot.com/2018/12...  
1  https://game-scripts-wiki.blogspot.com/2018/12...  
2  https://game-scripts-wiki.blogspot.com/2018/12...  
3  https://game-scripts-wiki.blogspot.com/2018/12...  
4  https://game-scripts-wiki.blogspot.com/2018/12...  
  game_id scene_id  character_id
0     re7  re7_003            67
1     re7  re7_004            67
2     re7  re7_009            67
3     re7  re7_020            67
4     re7  re7_023            67


In [30]:
game_id = "re8"
source = "https://game-scripts-wiki.blogspot.com/2021/05/resident-evil-8-village-full-transcript.html"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_8_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^CHAPTER\s+\d+:", re.IGNORECASE)
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Ethan": "Ethan Winters",
    "ETHAN": "Ethan Winters",
    "Mia": "Mia Winters",
    "MIA": "Mia Winters",
    "Chris": "Chris Redfield",
    "CHRIS": "Chris Redfield",
    "Rose": "Rosemary Winters",
    "ROSE": "Rosemary Winters",
    "Duke": "Duke",
    "DUKE": "Duke",
    "Mother Miranda": "Mother Miranda",
    "MOTHER MIRANDA": "Mother Miranda",
    "Lady Dimitrescu": "Alcina Dimitrescu",
    "LADY DIMITRESCU": "Alcina Dimitrescu",
    "Heisenberg": "Karl Heisenberg",
    "HEISENBERG": "Karl Heisenberg",
    "Moreau": "Salvatore Moreau",
    "MOREAU": "Salvatore Moreau",
    "Donna": "Donna Beneviento",
    "DONNA": "Donna Beneviento",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        title = line
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(10, 4)
(27, 3)
  game_id scene_id                                       title  \
0     re8  re8_001               CHAPTER 1: Almost Normal Life   
1     re8  re8_002                     CHAPTER 2: Cold Village   
2     re8  re8_003        CHAPTER 3: Road to the Famous Castle   
3     re8  re8_004                CHAPTER 4: Castle Dimitrescu   
4     re8  re8_005  CHAPTER 5: What The Hell Is Going On Here?   

                                              source  
0  https://game-scripts-wiki.blogspot.com/2021/05...  
1  https://game-scripts-wiki.blogspot.com/2021/05...  
2  https://game-scripts-wiki.blogspot.com/2021/05...  
3  https://game-scripts-wiki.blogspot.com/2021/05...  
4  https://game-scripts-wiki.blogspot.com/2021/05...  
  game_id scene_id  character_id
0     re8  re8_001            67
1     re8  re8_001            68
2     re8  re8_001            15
3     re8  re8_002            67
4     re8  re8_003            75


In [31]:
game_id = "re_sr"
source = "https://game-scripts-wiki.blogspot.com/2022/11/resident-evil-8-village-shadows-of-rose.html"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_8_rose_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^\[.+\]$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Rose": "Rosemary Winters",
    "ROSE": "Rosemary Winters",
    "K": "Nathan Dempsey",
    "Michael": "Michael",
    "Mia": "Mia Winters",
    "MIA": "Mia Winters",
    "Duke": "Duke",
    "DUKE": "Duke",
    "Ethan": "Ethan Winters",
    "ETHAN": "Ethan Winters",
    "Miranda": "Mother Miranda",
    "MIRANDA": "Mother Miranda",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        title = line.strip("[]").strip()
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(248, 4)
(199, 3)
  game_id   scene_id                                              title  \
0   re_sr  re_sr_001  A man wearing glasses and a leather jacket sit...   
1   re_sr  re_sr_002                     Rose looks at him in surprise.   
2   re_sr  re_sr_003                              They go into his lab.   
3   re_sr  re_sr_004  He points to a voluminous container with a red...   
4   re_sr  re_sr_005  She brings her hand to the container. The mush...   

                                              source  
0  https://game-scripts-wiki.blogspot.com/2022/11...  
1  https://game-scripts-wiki.blogspot.com/2022/11...  
2  https://game-scripts-wiki.blogspot.com/2022/11...  
3  https://game-scripts-wiki.blogspot.com/2022/11...  
4  https://game-scripts-wiki.blogspot.com/2022/11...  
  game_id   scene_id  character_id
0   re_sr  re_sr_001            74
1   re_sr  re_sr_001            82
2   re_sr  re_sr_002            74
3   re_sr  re_sr_002            82
4   re_sr  re_sr_003      

In [32]:
game_id = "re9"
source = "https://game-scripts-wiki.blogspot.com/2026/03/resident-evil-requiem-full-transcript.html"
path = "/kaggle/input/datasets/tommasofacchin/resident-evil-dialogues/Resident_evil_9_dialogues.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

scene_line_re = re.compile(r"^\[.+\]$")
dialogue_line_re = re.compile(r"^([^:\[]+?):\s*(.+)$")

scene_rows = []
scene_chars = {}
current_scene_id = None
scene_counter = 0

known_characters = {
    "Leon": "Leon Scott Kennedy",
    "Sherry": "Sherry Birkin",
    "Grace": "Grace Ashcroft",
    "Nathan": "Nathan Dempsey",
    "Victor": "Victor Gideon",
    "Alyssa": "Alyssa Ashcroft",
}

char_name_to_id = dict(zip(characters["name"], characters["id"]))
canon_to_id = {
    canon_name: char_name_to_id[canon_name]
    for canon_name in set(known_characters.values())
    if canon_name in char_name_to_id
}

for raw in lines:
    line = raw.strip()
    if not line:
        continue

    m_scene = scene_line_re.match(line)
    if m_scene:
        scene_counter += 1
        scene_id = f"{game_id}_{scene_counter:03d}"
        current_scene_id = scene_id

        title = line.strip("[]").strip()
        scene_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "title": title,
            "source": source,
        })
        scene_chars[scene_id] = set()
        continue

    if current_scene_id is not None:
        m_dialogue = dialogue_line_re.match(line)
        if m_dialogue:
            speaker_raw = m_dialogue.group(1).strip()
            speaker_base = speaker_raw.split("(")[0].strip()
            speaker_key = speaker_base.strip(" *[]").strip()

            canon_name = known_characters.get(speaker_key)
            if canon_name and canon_name in canon_to_id:
                scene_chars[current_scene_id].add(canon_to_id[canon_name])

scenes_new = pd.DataFrame(scene_rows)
scenes = pd.concat([scenes, scenes_new], ignore_index=True)

interaction_rows = []
for scene_id, ids in scene_chars.items():
    for cid in ids:
        interaction_rows.append({
            "game_id": game_id,
            "scene_id": scene_id,
            "character_id": cid,
        })

interactions_new = pd.DataFrame(interaction_rows)
interactions = pd.concat([interactions, interactions_new], ignore_index=True)

print(scenes_new.shape)
print(interactions_new.shape)
print(scenes_new.head())
print(interactions_new.head())

(646, 4)
(548, 3)
  game_id scene_id                                              title  \
0     re9  re9_001  We arrive at the threshold of the Federal Bure...   
1     re9  re9_002  He knocks on the file cabinet to get her atten...   
2     re9  re9_003  She clumsily takes a folder of documents, drop...   
3     re9  re9_004  She rushes into an office where her boss is al...   
4     re9  re9_005                             He hands her a folder.   

                                              source  
0  https://game-scripts-wiki.blogspot.com/2026/03...  
1  https://game-scripts-wiki.blogspot.com/2026/03...  
2  https://game-scripts-wiki.blogspot.com/2026/03...  
3  https://game-scripts-wiki.blogspot.com/2026/03...  
4  https://game-scripts-wiki.blogspot.com/2026/03...  
  game_id scene_id  character_id
0     re9  re9_001            82
1     re9  re9_002            80
2     re9  re9_002            82
3     re9  re9_003            80
4     re9  re9_004            80


In [33]:
scenes.to_csv("scenes.csv", index=False)
interactions.to_csv("interactions.csv", index=False)
print(scenes.shape)
print(interactions.shape)

(1573, 4)
(1537, 3)


In [34]:
inter_char = interactions.merge(
    characters,
    left_on="character_id",
    right_on="id",
    how="left"
)

inter_unique = inter_char.drop_duplicates(
    subset=["game_id", "scene_id", "character_id"]
)

scene_counts = (
    inter_unique
    .groupby(["game_id", "character_id", "name"], as_index=False)
    .size()
    .rename(columns={"size": "num_scenes"})
    .sort_values(["game_id", "num_scenes"], ascending=[True, False])
)

for game in scene_counts["game_id"].unique():
    print(game)
    sub = scene_counts[scene_counts["game_id"] == game]
    for _, row in sub.iterrows():
        print(f" - {row['name']} (id={row['character_id']}) : {row['num_scenes']} scenes")
    print()

re
 - Chris Redfield (id=15) : 16 scenes
 - Jill Valentine (id=16) : 16 scenes
 - Albert Wesker (id=6) : 12 scenes
 - Barry Burton (id=17) : 10 scenes
 - Rebecca Chambers (id=20) : 8 scenes
 - Brad Vickers (id=18) : 7 scenes

re0
 - Rebecca Chambers (id=20) : 34 scenes
 - Billy Coen (id=42) : 25 scenes
 - Enrico Marini (id=22) : 4 scenes
 - Edward Dewey (id=44) : 2 scenes
 - James Marcus (id=43) : 1 scenes

re2
 - Leon Scott Kennedy (id=1) : 49 scenes
 - Claire Redfield (id=25) : 48 scenes
 - Sherry Birkin (id=26) : 30 scenes
 - Ada Wong (id=3) : 29 scenes
 - Annette Birkin (id=28) : 12 scenes
 - William Birkin (id=27) : 5 scenes
 - Ben Bertolucci (id=31) : 4 scenes

re3
 - Jill Valentine (id=16) : 42 scenes
 - Mikhail Victor (id=36) : 4 scenes
 - Brad Vickers (id=18) : 3 scenes
 - Nemesis (id=34) : 3 scenes

re4
 - Leon Scott Kennedy (id=1) : 16 scenes
 - Ashley Graham (id=2) : 9 scenes
 - Ada Wong (id=3) : 5 scenes
 - Lord Osmund Saddler (id=11) : 5 scenes
 - Luis Sera (id=5) : 4 sce

In [35]:
print(f"\n=============================\ngames table: \n=============================\n{games.describe}")
print(f"\n\n=============================\n characters table: \n=============================\n{characters.describe}")
print(f"\n\n=============================\n gameAppearance table: \n=============================\n{gameAppearance.describe}")
print(f"\n\n=============================\n scenes table: \n=============================\n{scenes.describe}")
print(f"\n\n=============================\n interactions table: \n=============================\n{interactions.describe}")


games table: 
<bound method NDFrame.describe of         id                                   title  year      type  \
0       re                           Resident Evil  1996  mainline   
1      re2                         Resident Evil 2  1998  mainline   
2      re3                Resident Evil 3: Nemesis  1999  mainline   
3    re_ve          Resident Evil: Code – Veronica  2000  mainline   
4      re0                         Resident Evil 0  2002  mainline   
5      re4                         Resident Evil 4  2005  mainline   
6      re5                         Resident Evil 5  2009  mainline   
7    re_re              Resident Evil: Revelations  2012   spinoff   
8      re6                         Resident Evil 6  2012  mainline   
9   re_re2            Resident Evil: Revelations 2  2015   spinoff   
10     re7              Resident Evil 7: Biohazard  2017  mainline   
11     re8                   Resident Evil Village  2021  mainline   
12   re_sr  Resident Evil Village – Shado